In [1]:
!pip -q install kagglehub gradio pandas numpy scikit-learn pypdf plotly
print("All libraries installed successfully!")


All libraries installed successfully!


In [2]:

import os
import re
import glob
import tempfile
import numpy as np
import pandas as pd
import gradio as gr
import kagglehub
import plotly.express as px

from pypdf import PdfReader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:

print("Loading Resume Dataset...")

try:
    dataset_path = kagglehub.dataset_download(
        "snehaanbhawal/resume-dataset"
    )

    csv_files = glob.glob(
        os.path.join(dataset_path, "**", "*.csv"),
        recursive=True
    )

    if not csv_files:
        raise FileNotFoundError("CSV not found")

    df = pd.read_csv(csv_files[0])
    print("Kaggle dataset downloaded successfully!")

except Exception as e:
    print("Kaggle download failed:", str(e))
    print("Creating fallback dataset...")

    samples = {
        "Data Science": [
            "Python machine learning pandas numpy SQL statistics",
            "Deep learning TensorFlow data analytics Python",
            "Machine learning predictive analytics scikit-learn"
        ],
        "Data Analyst": [
            "Excel SQL Power BI Tableau data visualization",
            "Python pandas reporting business intelligence",
            "SQL data analysis statistics dashboard"
        ],
        "AI Engineer": [
            "Python artificial intelligence machine learning NLP",
            "LLM generative AI deep learning TensorFlow",
            "PyTorch neural networks NLP Python"
        ],
        "Web Developer": [
            "HTML CSS JavaScript React frontend",
            "Node.js React JavaScript web development",
            "Python Django Flask web application"
        ],
        "Java Developer": [
            "Java Spring Boot SQL OOP programming",
            "Java Hibernate collections multithreading",
            "Java backend application software development"
        ]
    }

    records = []

    for category, resumes in samples.items():
        for i in range(100):
            records.append({
                "Resume_str": resumes[i % len(resumes)],
                "Category": category
            })

    df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
display(df.head())


Loading Resume Dataset...
Using Colab cache for faster access to the 'resume-dataset' dataset.
Kaggle dataset downloaded successfully!
Dataset shape: (2484, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:

df.columns = [c.strip() for c in df.columns]

text_col = next(
    (c for c in ["Resume_str", "resume_text", "Resume"]
     if c in df.columns),
    None
)

category_col = next(
    (c for c in ["Category", "category"]
     if c in df.columns),
    None
)

if text_col is None or category_col is None:
    raise ValueError("Required dataset columns not found!")

df = df[[text_col, category_col]].dropna().copy()
df.columns = ["resume", "category"]

df["resume"] = df["resume"].astype(str)
df["category"] = df["category"].astype(str)

df["resume"] = df["resume"].str.lower()
df["resume"] = df["resume"].str.replace(
    r"[^a-z0-9+#.\s]", " ", regex=True
)

print("Preprocessing completed!")
print("Total resumes:", len(df))
print("Total categories:", df["category"].nunique())

display(df["category"].value_counts().head(15))


Preprocessing completed!
Total resumes: 2484
Total categories: 24


,count
category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,120
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,118
FITNESS,117
AVIATION,117


In [5]:

category_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X = category_vectorizer.fit_transform(df["resume"])

encoder = LabelEncoder()
y = encoder.fit_transform(df["category"])

category_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

category_model.fit(X, y)

print("Resume classification model trained!")

print("Number of categories:", len(encoder.classes_))
print("Categories:", list(encoder.classes_[:20]))


Resume classification model trained!
Number of categories: 24
Categories: ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR']


In [6]:

SKILLS = [
    "python", "java", "c++", "sql", "mysql", "mongodb",
    "machine learning", "deep learning",
    "artificial intelligence", "nlp", "generative ai",
    "llm", "tensorflow", "pytorch", "scikit-learn",
    "pandas", "numpy", "matplotlib",
    "data analysis", "data visualization",
    "power bi", "tableau", "excel", "statistics",
    "aws", "azure", "docker", "git", "github",
    "html", "css", "javascript", "react", "node.js",
    "flask", "django", "fastapi", "rest api",
    "communication", "problem solving",
    "data structures", "algorithms", "linux"
]


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_skills(text):
    text = clean_text(text)
    found = []

    for skill in SKILLS:
        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(skill)
            + r"(?![a-z0-9])"
        )

        if re.search(pattern, text):
            found.append(skill)

    return sorted(set(found))


def extract_pdf_text(file_path):
    try:
        reader = PdfReader(file_path)

        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    except Exception as e:
        print("PDF extraction error:", e)
        return ""


def extract_experience(text):
    patterns = [
        r"(\d+)\+?\s+years?\s+of\s+experience",
        r"(\d+)\+?\s+years?\s+experience",
        r"(\d+)\+?\s+years?\s+in"
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text.lower())

        if matches:
            return max(map(int, matches))

    return 0


def extract_education(text):
    education = [
        "phd", "doctorate", "master", "mba",
        "bachelor", "b.tech", "b.e", "bsc",
        "m.tech", "m.sc", "degree"
    ]

    text = text.lower()

    return any(word in text for word in education)


print("Resume extraction and skill detection functions ready!")


Resume extraction and skill detection functions ready!


In [7]:

def screen_resume(resume_text, job_description, preferred_skills):

    resume_clean = clean_text(resume_text)
    jd_clean = clean_text(job_description)

    required = extract_skills(job_description)
    preferred = extract_skills(preferred_skills)

    candidate_skills = extract_skills(resume_text)

    matched = sorted(set(required) & set(candidate_skills))
    missing = sorted(set(required) - set(candidate_skills))

    preferred_matched = sorted(
        set(preferred) & set(candidate_skills)
    )

    # Required skills: 50 points
    required_score = (
        50 * len(matched) / len(required)
        if required else 0
    )

    # Preferred skills: 15 points
    preferred_score = (
        15 * len(preferred_matched) / len(preferred)
        if preferred else 0
    )

    # Semantic similarity: 20 points
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    vectors = vectorizer.fit_transform([
        jd_clean, resume_clean
    ])

    similarity = cosine_similarity(
        vectors[0:1], vectors[1:2]
    )[0][0]

    similarity_score = similarity * 20

    # Experience: 10 points
    required_exp = extract_experience(job_description)
    candidate_exp = extract_experience(resume_text)

    if required_exp > 0:
        experience_score = min(
            candidate_exp / required_exp, 1
        ) * 10
    else:
        experience_score = 10

    # Education: 5 points
    education_score = (
        5 if extract_education(resume_text) else 0
    )

    total = round(min(
        required_score +
        preferred_score +
        similarity_score +
        experience_score +
        education_score,
        100
    ), 2)

    # Resume category prediction
    category_input = category_vectorizer.transform(
        [resume_clean]
    )

    predicted_class = category_model.predict(
        category_input
    )[0]

    category = encoder.inverse_transform(
        [predicted_class]
    )[0]

    probabilities = category_model.predict_proba(
        category_input
    )[0]

    confidence = round(float(max(probabilities)) * 100, 2)

    # Explain similarity through overlapping terms
    features = vectorizer.get_feature_names_out()

    contributions = (
        vectors[0].toarray()[0] *
        vectors[1].toarray()[0]
    )

    indices = np.argsort(contributions)[::-1][:8]

    important_terms = [
        features[i] for i in indices
        if contributions[i] > 0
    ]

    if total >= 75:
        recommendation = "Strong Match"
    elif total >= 50:
        recommendation = "Moderate Match"
    else:
        recommendation = "Low Match"

    return {
        "score": total,
        "required": round(required_score, 2),
        "preferred": round(preferred_score, 2),
        "similarity": round(similarity_score, 2),
        "experience": round(experience_score, 2),
        "education": round(education_score, 2),
        "matched": matched,
        "missing": missing,
        "preferred_matched": preferred_matched,
        "skills": candidate_skills,
        "experience_years": candidate_exp,
        "required_experience": required_exp,
        "category": category,
        "confidence": confidence,
        "terms": important_terms,
        "recommendation": recommendation
    }


print("Explainable AI screening engine is ready!")


Explainable AI screening engine is ready!


In [8]:

def analyze_candidates(files, job_description, preferred_skills):

    if not files:
        raise gr.Error("Please upload PDF resumes.")

    if not job_description.strip():
        raise gr.Error("Please enter a job description.")

    results = []

    for file_path in files:

        resume_text = extract_pdf_text(file_path)

        if len(resume_text.strip()) < 30:
            continue

        analysis = screen_resume(
            resume_text,
            job_description,
            preferred_skills
        )

        results.append({
            "Candidate": os.path.basename(file_path),
            "Match Score": analysis["score"],
            "Category": analysis["category"],
            "Confidence": analysis["confidence"],
            "Required Skills": analysis["required"],
            "Preferred Skills": analysis["preferred"],
            "Similarity": analysis["similarity"],
            "Experience": analysis["experience"],
            "Education": analysis["education"],
            "Recommendation": analysis["recommendation"],
            "Analysis": analysis
        })

    if not results:
        raise gr.Error("No readable PDF resumes found.")

    results.sort(
        key=lambda x: x["Match Score"],
        reverse=True
    )

    table = pd.DataFrame([
        {k: v for k, v in r.items() if k != "Analysis"}
        for r in results
    ])

    explanation = "# EXPLAINHIRE - AI SCREENING REPORT\n"

    for rank, result in enumerate(results, 1):

        a = result["Analysis"]

        explanation += f"""

---

## {rank}. {result['Candidate']}

**Match Score: {a['score']} / 100**

**Recommendation:** {a['recommendation']}

**Predicted Category:** {a['category']}

**Category Model Confidence:** {a['confidence']}%

### Score Breakdown

| Component | Score |
|---|---:|
| Required skills | {a['required']} / 50 |
| Preferred skills | {a['preferred']} / 15 |
| Semantic similarity | {a['similarity']} / 20 |
| Experience | {a['experience']} / 10 |
| Education | {a['education']} / 5 |

### Explainable AI Analysis

**Matched Skills:**
{', '.join(a['matched']) or 'None identified'}

**Missing Skills:**
{', '.join(a['missing']) or 'None identified'}

**Preferred Skills Matched:**
{', '.join(a['preferred_matched']) or 'None identified'}

**Detected Candidate Skills:**
{', '.join(a['skills']) or 'None identified'}

**Important Matching Terms:**
{', '.join(a['terms']) or 'None identified'}

**Candidate Experience:** {a['experience_years']} years

**Required Experience:** {a['required_experience']} years

"""

    explanation += """

---

**Note:** These scores are decision-support indicators.
A human recruiter must review each candidate. They are
not automatic hiring or rejection decisions.
"""

    # Export CSV
    export_path = os.path.join(
        tempfile.gettempdir(),
        "ExplainHire_Candidate_Ranking.csv"
    )

    table.to_csv(export_path, index=False)

    # Candidate chart
    fig = px.bar(
        table,
        x="Candidate",
        y="Match Score",
        color="Match Score",
        color_continuous_scale=["#0B3B50", "#08B8D4"],
        range_y=[0, 100],
        title="Candidate Match Analysis"
    )

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="#101827",
        plot_bgcolor="#101827",
        font_color="white"
    )

    return table, explanation, fig, export_path


print("Candidate ranking and report generation ready!")


Candidate ranking and report generation ready!


In [9]:

css = """
body {
    background: #0b1120 !important;
}

.gradio-container {
    max-width: 1400px !important;
    background: #0b1120 !important;
}

.hero {
    background: linear-gradient(120deg,#111c35,#073a50);
    border: 1px solid #087d9b;
    border-radius: 18px;
    padding: 28px;
    margin-bottom: 15px;
}

.hero h1 {
    font-size: 38px;
    color: #39d5ef;
}

.hero p {
    color: #b9c8d9;
}

"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        neutral_hue="slate"
    ),
    css=css,
    title="ExplainHire AI"
) as demo:

    gr.HTML("""
    <div class="hero">
        <h1>✦ ExplainHire</h1>
        <h2>Intelligent Resume Screening</h2>
        <p>
        Explainable AI Framework for Transparent
        Candidate Ranking and Recruitment Analytics
        </p>
        <p>
        NLP | Machine Learning | Explainable AI | Analytics
        </p>
    </div>
    """)

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("## 01 — Job Requirements")

            job_description = gr.Textbox(
                label="Job Description",
                placeholder=(
                    "Example: AI/ML Engineer with Python, "
                    "Machine Learning, SQL, TensorFlow "
                    "and 2 years experience."
                ),
                lines=8
            )

            preferred_skills = gr.Textbox(
                label="Preferred Skills",
                placeholder="AWS, Docker, Git, NLP",
                lines=3
            )

            resume_files = gr.File(
                label="Upload Multiple PDF Resumes",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            analyze_button = gr.Button(
                "✦ Analyze & Rank Candidates",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=2):

            gr.Markdown("## 02 — Candidate Analytics")

            candidate_table = gr.Dataframe(
                label="Candidate Ranking",
                interactive=False
            )

            score_chart = gr.Plot(
                label="Match Score Visualization"
            )

    with gr.Accordion(
        "03 — Explainable AI Detailed Report",
        open=True
    ):

        report = gr.Markdown()

    csv_download = gr.File(
        label="Download Candidate Ranking CSV"
    )

    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            resume_files,
            job_description,
            preferred_skills
        ],
        outputs=[
            candidate_table,
            report,
            score_chart,
            csv_download
        ]
    )

    gr.HTML("""
    <div style="
        text-align:center;
        padding:25px;
        color:#8294a9;
        border-top:1px solid #243449;
        margin-top:30px;">
        EXPLAINHIRE | AI RECRUITMENT INTELLIGENCE
        <br>
        Academic Research Prototype
    </div>
    """)

print("Dashboard interface created successfully!")


/tmp/ipykernel_888/1046896644.py:30: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Dashboard interface created successfully!


In [10]:

demo.launch(
    share=True,
    debug=False
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2afdeedfa66ca1b55e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:

!pip -q install edge-tts nest_asyncio

import asyncio
import edge_tts
import nest_asyncio

nest_asyncio.apply()

print("Voice libraries installed!")


Voice libraries installed!


In [12]:

candidate_results = []

def analyze_candidates(files, job_description, preferred_skills):

    global candidate_results

    if not files:
        raise gr.Error("Upload PDF resumes.")

    if not job_description.strip():
        raise gr.Error("Enter a job description.")

    results = []

    for file_path in files:

        resume_text = extract_pdf_text(file_path)

        if len(resume_text.strip()) < 30:
            continue

        analysis = screen_resume(
            resume_text,
            job_description,
            preferred_skills
        )

        results.append({
            "Candidate": os.path.basename(file_path),
            "Match Score": analysis["score"],
            "Category": analysis["category"],
            "Confidence": analysis["confidence"],
            "Required Skills": analysis["required"],
            "Preferred Skills": analysis["preferred"],
            "Similarity": analysis["similarity"],
            "Experience": analysis["experience"],
            "Education": analysis["education"],
            "Recommendation": analysis["recommendation"],
            "Analysis": analysis
        })

    if not results:
        raise gr.Error("No readable resumes found.")

    results.sort(
        key=lambda x: x["Match Score"],
        reverse=True
    )

    candidate_results = results

    table = pd.DataFrame([
        {k: v for k, v in r.items() if k != "Analysis"}
        for r in results
    ])

    report = "# EXPLAINHIRE - CANDIDATE REPORT\n"

    for i, r in enumerate(results, 1):

        a = r["Analysis"]

        report += f"""
---
## {i}. {r['Candidate']}

**Score:** {a['score']}/100

**Category:** {a['category']}

**Recommendation:** {a['recommendation']}

**Matched Skills:** {', '.join(a['matched'])}

**Missing Skills:** {', '.join(a['missing'])}

**Experience:** {a['experience_years']} years

"""

    export_path = "/content/ExplainHire_Ranking.csv"
    table.to_csv(export_path, index=False)

    fig = px.bar(
        table,
        x="Candidate",
        y="Match Score",
        color="Match Score",
        range_y=[0, 100],
        title="Candidate Ranking"
    )

    fig.update_layout(template="plotly_dark")

    choices = [r["Candidate"] for r in results]

    return (
        table,
        report,
        fig,
        export_path,
        gr.update(choices=choices, value=choices[0])
    )


print("Candidate analysis and voice integration ready!")


Candidate analysis and voice integration ready!


In [13]:

async def generate_candidate_voice(candidate_name, voice_choice):

    candidate = next(
        (r for r in candidate_results
         if r["Candidate"] == candidate_name),
        None
    )

    if candidate is None:
        raise gr.Error("Candidate not found.")

    a = candidate["Analysis"]

    voice_map = {
        "Male": "en-IN-PrabhatNeural",
        "Female": "en-IN-NeerjaNeural"
    }

    voice = voice_map[voice_choice]

    narration = f"""
    Welcome to ExplainHire candidate analysis.

    Candidate name: {candidate_name}.

    The overall resume matching score is
    {a['score']} out of 100.

    The predicted job category is {a['category']}.

    The recommendation is {a['recommendation']}.

    The candidate has matched the following required skills:
    {', '.join(a['matched']) or 'No required skills identified'}.

    The following required skills are missing:
    {', '.join(a['missing']) or 'No missing required skills'}.

    The candidate has {a['experience_years']} years
    of detected experience.

    The required experience is
    {a['required_experience']} years.

    The required skills score is {a['required']} out of 50.

    The preferred skills score is {a['preferred']} out of 15.

    The similarity score is {a['similarity']} out of 20.

    The experience score is {a['experience']} out of 10.

    The education score is {a['education']} out of 5.

    This concludes the candidate analysis.
    A recruiter should review the candidate before
    making a hiring decision.
    """

    output_path = "/content/ExplainHire_Voice.mp3"

    communicate = edge_tts.Communicate(
        narration,
        voice
    )

    await communicate.save(output_path)

    return output_path


print("Voice narration function ready!")


Voice narration function ready!


In [14]:

css = """
body {
    background: #0b1120 !important;
}

.gradio-container {
    max-width: 1400px !important;
    background: #0b1120 !important;
}

.hero {
    background: linear-gradient(120deg,#111c35,#073a50);
    border: 1px solid #087d9b;
    border-radius: 18px;
    padding: 28px;
}

.hero h1 {
    color: #39d5ef;
    font-size: 38px;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        neutral_hue="slate"
    ),
    css=css,
    title="ExplainHire AI"
) as demo:

    gr.HTML("""
    <div class="hero">
        <h1>✦ ExplainHire</h1>
        <h2>Intelligent Resume Screening</h2>
        <p>Explainable AI | NLP | Candidate Ranking | Voice Assistant</p>
    </div>
    """)

    with gr.Row():

        with gr.Column():

            gr.Markdown("## Job Requirements")

            job_description = gr.Textbox(
                label="Job Description",
                lines=8
            )

            preferred_skills = gr.Textbox(
                label="Preferred Skills"
            )

            resume_files = gr.File(
                label="Upload PDF Resumes",
                file_count="multiple",
                file_types=[".pdf"],
                type="filepath"
            )

            analyze_button = gr.Button(
                "✦ Analyze Candidates",
                variant="primary"
            )

        with gr.Column():

            gr.Markdown("## Candidate Analytics")

            candidate_table = gr.Dataframe(
                label="Candidate Ranking",
                interactive=False
            )

            score_chart = gr.Plot()

    with gr.Accordion(
        "Explainable AI Report",
        open=True
    ):
        report = gr.Markdown()

    csv_download = gr.File(
        label="Download CSV Report"
    )

    gr.Markdown("---")
    gr.Markdown("# 🎙 ExplainHire Voice Assistant")

    gr.Markdown(
        "Listen to the individual candidate's "
        "AI-generated screening explanation."
    )

    candidate_selector = gr.Dropdown(
        choices=[],
        label="Select Candidate",
        interactive=True
    )

    voice_selector = gr.Radio(
        choices=["Male", "Female"],
        value="Female",
        label="Narration Voice"
    )

    voice_button = gr.Button(
        "🔊 Generate Voice Explanation",
        variant="primary"
    )

    voice_audio = gr.Audio(
        label="Candidate Explanation Audio",
        type="filepath"
    )

    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            resume_files,
            job_description,
            preferred_skills
        ],
        outputs=[
            candidate_table,
            report,
            score_chart,
            csv_download,
            candidate_selector
        ]
    )

    voice_button.click(
        fn=generate_candidate_voice,
        inputs=[
            candidate_selector,
            voice_selector
        ],
        outputs=voice_audio
    )

    gr.HTML("""
    <div style="
        text-align:center;
        padding:25px;
        color:#8294a9;">
        EXPLAINHIRE | AI RECRUITMENT INTELLIGENCE
        <br>
        Explainable AI Voice Assistant
    </div>
    """)

print("Updated dashboard with voice assistant!")


Updated dashboard with voice assistant!


/tmp/ipykernel_888/3876090280.py:24: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



In [15]:

demo.launch(
    share=True,
    debug=False
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://150dc3707ba56f6d15.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:

# CELL 14A — USER AUTHENTICATION

import sqlite3
import hashlib
import secrets
import os

DB_PATH = "/content/explainhire_users.db"

def create_user_database():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            salt TEXT NOT NULL,
            role TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()


def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        salt.encode(),
        100000
    ).hex()


def register_user(name, email, password, role):
    name = name.strip()
    email = email.strip().lower()

    if not name or not email or not password:
        return "Please fill in all fields."

    if len(password) < 8:
        return "Password must contain at least 8 characters."

    if role not in ["Recruiter", "Candidate"]:
        return "Invalid account type."

    salt = secrets.token_hex(16)
    password_hash = hash_password(password, salt)

    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()

        cursor.execute("""
            INSERT INTO users
            (name, email, password_hash, salt, role)
            VALUES (?, ?, ?, ?, ?)
        """, (name, email, password_hash, salt, role))

        conn.commit()
        conn.close()

        return "Account created successfully. You can now log in."

    except sqlite3.IntegrityError:
        return "This email is already registered."


def login_user(email, password, role):
    email = email.strip().lower()

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT name, email, password_hash, salt, role
        FROM users WHERE email = ?
    """, (email,))

    user = cursor.fetchone()
    conn.close()

    if not user:
        return None, "Account not found."

    name, saved_email, saved_hash, salt, saved_role = user

    if saved_role != role:
        return None, "Please select the correct account type."

    entered_hash = hash_password(password, salt)

    if not secrets.compare_digest(entered_hash, saved_hash):
        return None, "Incorrect password."

    return {
        "name": name,
        "email": saved_email,
        "role": saved_role
    }, "Login successful."


create_user_database()

print("ExplainHire authentication system is ready.")


ExplainHire authentication system is ready.


In [17]:

# CELL 14B — CANDIDATE DASHBOARD FUNCTION

from pypdf import PdfReader

def candidate_resume_review(pdf_file, job_description):

    if pdf_file is None:
        return "Please upload your resume.", "", ""

    try:
        reader = PdfReader(pdf_file)
        resume_text = " ".join(
            page.extract_text() or "" for page in reader.pages
        )

        if not resume_text.strip():
            return "No readable text found in the PDF.", "", ""

        candidate_skills = set(extract_skills(resume_text))

        if not job_description.strip():
            skills_text = ", ".join(sorted(candidate_skills))
            return (
                "Resume uploaded successfully.",
                skills_text or "No recognized skills found.",
                "Enter a job description to check your skill match."
            )

        job_skills = set(extract_skills(job_description))

        matched = sorted(candidate_skills.intersection(job_skills))
        missing = sorted(job_skills.difference(candidate_skills))

        if job_skills:
            match_percentage = round(
                len(matched) / len(job_skills) * 100, 2
            )
        else:
            match_percentage = 0

        report = f"""
        RESUME REVIEW

        Skill Match: {match_percentage}%

        Matched Skills:
        {", ".join(matched) if matched else "None"}

        Skills to Improve:
        {", ".join(missing) if missing else "None"}
        """

        return (
            "Resume analyzed successfully.",
            ", ".join(sorted(candidate_skills)) or "No recognized skills found.",
            report
        )

    except Exception as e:
        return f"Error reading resume: {str(e)}", "", ""


In [19]:

# CELL 14C — COMPLETE EXPLAINHIRE LOGIN APP

import gradio as gr

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    title="ExplainHire | AI Resume Screening"
) as demo:

    session = gr.State(None)

    gr.Markdown("""
    # ExplainHire
    ### Intelligent Resume Screening & Candidate Ranking
    Secure access for recruiters and candidates.
    """)

    # ---------------- LOGIN / REGISTER ----------------

    auth_panel = gr.Column(visible=True)

    with auth_panel:

        with gr.Tabs():

            with gr.Tab("Login"):

                gr.Markdown("### Welcome Back")

                login_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Recruiter",
                    label="Login as"
                )

                login_email = gr.Textbox(
                    label="Email",
                    placeholder="Enter your email"
                )

                login_password = gr.Textbox(
                    label="Password",
                    type="password"
                )

                login_button = gr.Button(
                    "Login",
                    variant="primary"
                )

                login_status = gr.Markdown()

            with gr.Tab("Create Account"):

                gr.Markdown("### Join ExplainHire")

                register_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Candidate",
                    label="Create account as"
                )

                register_name = gr.Textbox(
                    label="Full Name"
                )

                register_email = gr.Textbox(
                    label="Email"
                )

                register_password = gr.Textbox(
                    label="Password",
                    type="password"
                )

                register_button = gr.Button(
                    "Create Account",
                    variant="primary"
                )

                register_status = gr.Markdown()

    # ---------------- RECRUITER DASHBOARD ----------------

    recruiter_panel = gr.Column(visible=False)

    with recruiter_panel:

        gr.Markdown("""
        ## Recruiter Dashboard
        Screen resumes, rank candidates and understand why they match.
        """)

        recruiter_welcome = gr.Markdown()

        recruiter_files = gr.File(
            label="Upload Candidate Resumes",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        )

        recruiter_jd = gr.Textbox(
            label="Job Description",
            lines=7,
            placeholder="Paste the job description here..."
        )

        recruiter_required = gr.Textbox(
            label="Required Skills",
            placeholder="Python, Machine Learning, SQL"
        )

        recruiter_preferred = gr.Textbox(
            label="Preferred Skills",
            placeholder="NLP, LLM, AWS, Docker"
        )

        recruiter_experience = gr.Number(
            label="Minimum Experience (Years)",
            value=0,
            minimum=0
        )

        analyze_button = gr.Button(
            "Analyze Resumes",
            variant="primary"
        )

        recruiter_table = gr.Dataframe(
            label="Candidate Ranking"
        )

        recruiter_report = gr.Markdown(
            label="Screening Report"
        )

        recruiter_plot = gr.Plot(
            label="Candidate Comparison"
        )

        recruiter_csv = gr.File(
            label="Download Screening Results"
        )

        gr.Markdown("### Voice Explanation")

        voice_candidate = gr.Dropdown(
            choices=[],
            label="Select Candidate"
        )

        voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Explanation Voice"
        )

        voice_button = gr.Button(
            "Generate Voice Explanation"
        )

        voice_output = gr.Audio(
            label="Candidate Explanation",
            type="filepath"
        )

        recruiter_logout = gr.Button("Logout")

    # ---------------- CANDIDATE DASHBOARD ----------------

    candidate_panel = gr.Column(visible=False)

    with candidate_panel:

        gr.Markdown("""
        ## Candidate Dashboard
        Upload your resume and review how your skills match a job.
        """)

        candidate_welcome = gr.Markdown()

        candidate_pdf = gr.File(
            label="Upload Your Resume",
            file_types=[".pdf"],
            type="filepath"
        )

        candidate_jd = gr.Textbox(
            label="Target Job Description",
            lines=7,
            placeholder="Paste the job description here..."
        )

        candidate_analyze = gr.Button(
            "Analyze My Resume",
            variant="primary"
        )

        candidate_status = gr.Markdown()

        candidate_skills = gr.Textbox(
            label="Skills Found in Your Resume",
            lines=3
        )

        candidate_report = gr.Textbox(
            label="Resume Match Report",
            lines=10
        )

        candidate_logout = gr.Button("Logout")

    # ---------------- REGISTRATION ----------------

    register_button.click(
        fn=register_user,
        inputs=[
            register_name,
            register_email,
            register_password,
            register_role
        ],
        outputs=register_status
    )

    # ---------------- LOGIN ----------------

    def authenticate(email, password, role):

        user, message = login_user(email, password, role)

        if user is None:
            return (
                None,
                message,
                gr.update(),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update()
            )

        if role == "Recruiter":
            return (
                user,
                message,
                gr.update(value=f"Welcome, **{user['name']}**"),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update()
            )

        return (
            user,
            message,
            gr.update(),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(value=f"Welcome, **{user['name']}**")
        )

    login_button.click(
        fn=authenticate,
        inputs=[
            login_email,
            login_password,
            login_role
        ],
        outputs=[
            session,
            login_status,
            recruiter_welcome,
            auth_panel,
            recruiter_panel,
            candidate_panel,
            candidate_welcome
        ]
    )

    # ---------------- RECRUITER SCREENING ----------------

    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            recruiter_files,
            recruiter_jd,
            recruiter_required,
            recruiter_preferred,
            recruiter_experience
        ],
        outputs=[
            recruiter_table,
            recruiter_report,
            recruiter_plot,
            recruiter_csv,
            voice_candidate
        ]
    )

    # ---------------- VOICE GENERATION ----------------

    voice_button.click(
        fn=generate_candidate_voice,
        inputs=[
            voice_candidate,
            voice_choice
        ],
        outputs=voice_output
    )

    # ---------------- CANDIDATE ANALYSIS ----------------

    candidate_analyze.click(
        fn=candidate_resume_review,
        inputs=[
            candidate_pdf,
            candidate_jd
        ],
        outputs=[
            candidate_status,
            candidate_skills,
            candidate_report
        ]
    )

    # ---------------- LOGOUT ----------------

    def logout_user():
        return (
            None,
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False)
        )

    recruiter_logout.click(
        fn=logout_user,
        outputs=[
            session,
            auth_panel,
            recruiter_panel,
            candidate_panel
        ]
    )

    candidate_logout.click(
        fn=logout_user,
        outputs=[
            session,
            auth_panel,
            recruiter_panel,
            candidate_panel
        ]
    )

print("ExplainHire app interface created.")


/tmp/ipykernel_888/3411511548.py:5: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning:

Expected 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1387: UserWarning:

Expected maximum 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.



ExplainHire app interface created.


In [20]:

# CELL 15 — LAUNCH APPLICATION

demo.launch(
    share=True,
    debug=False
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be881332a933154b09.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:

# CELL 14C — EXPLAINHIRE ANIMATED UI

import gradio as gr

custom_css = """

/* MAIN BACKGROUND */
.gradio-container {
    background:
      radial-gradient(circle at 10% 15%, #172554 0%, transparent 35%),
      radial-gradient(circle at 90% 80%, #3b0764 0%, transparent 35%),
      linear-gradient(135deg, #070b18, #10172c, #090d1c) !important;
    color: #f8fafc !important;
    font-family: 'Inter', sans-serif !important;
}

/* ANIMATED BACKGROUND */
.gradio-container:before {
    content: "";
    position: fixed;
    width: 350px;
    height: 350px;
    border-radius: 50%;
    background: #06b6d4;
    filter: blur(130px);
    opacity: .15;
    top: 5%;
    left: -100px;
    pointer-events: none;
    animation: floatOne 10s ease-in-out infinite alternate;
    z-index: 0;
}

.gradio-container:after {
    content: "";
    position: fixed;
    width: 350px;
    height: 350px;
    border-radius: 50%;
    background: #9333ea;
    filter: blur(130px);
    opacity: .17;
    bottom: 0;
    right: -100px;
    pointer-events: none;
    animation: floatTwo 12s ease-in-out infinite alternate;
    z-index: 0;
}

@keyframes floatOne {
    from { transform: translate(0, 0); }
    to { transform: translate(150px, 100px); }
}

@keyframes floatTwo {
    from { transform: translate(0, 0); }
    to { transform: translate(-130px, -100px); }
}

/* HEADER */
.hero {
    text-align: center;
    padding: 35px 10px 25px;
    animation: appear .9s ease-out;
}

.logo {
    font-size: 46px;
    display: inline-block;
    animation: pulse 2.5s infinite;
}

.hero h1 {
    font-size: 42px;
    font-weight: 850;
    letter-spacing: -1px;
    background: linear-gradient(90deg, #22d3ee, #a78bfa, #f0abfc);
    background-size: 200% auto;
    color: transparent;
    background-clip: text;
    -webkit-background-clip: text;
    animation: gradientMove 4s linear infinite;
}

.hero p {
    color: #a5b4fc;
    font-size: 15px;
}

@keyframes gradientMove {
    to { background-position: 200% center; }
}

@keyframes pulse {
    50% { transform: scale(1.12); }
}

@keyframes appear {
    from { opacity: 0; transform: translateY(20px); }
    to { opacity: 1; transform: translateY(0); }
}

/* LOGIN CARD */
#auth-panel {
    max-width: 480px !important;
    margin: 20px auto 50px !important;
    padding: 28px !important;
    background: rgba(17, 24, 48, .88) !important;
    border: 1px solid rgba(103, 232, 249, .3) !important;
    border-radius: 24px !important;
    box-shadow: 0 0 45px rgba(34, 211, 238, .12);
    backdrop-filter: blur(18px);
    animation: cardEnter .8s ease-out;
}

@keyframes cardEnter {
    from { opacity: 0; transform: translateY(35px) scale(.97); }
    to { opacity: 1; transform: translateY(0) scale(1); }
}

/* DASHBOARD PANELS */
.dashboard {
    background: rgba(15, 23, 42, .88) !important;
    border: 1px solid rgba(129, 140, 248, .25) !important;
    border-radius: 20px !important;
    padding: 22px !important;
    animation: appear .7s ease-out;
}

/* BUTTONS */
.gr-button {
    border-radius: 12px !important;
    font-weight: 700 !important;
    transition: all .25s ease !important;
    position: relative;
    overflow: hidden;
}

.gr-button-primary {
    background: linear-gradient(100deg, #0891b2, #6366f1, #9333ea) !important;
    border: none !important;
    color: white !important;
    box-shadow: 0 5px 20px rgba(99, 102, 241, .25);
}

.gr-button:hover {
    transform: translateY(-3px) scale(1.015) !important;
    box-shadow: 0 8px 28px rgba(34, 211, 238, .3) !important;
}

/* INPUTS */
input, textarea {
    background: rgba(15, 23, 42, .9) !important;
    border: 1px solid #334155 !important;
    border-radius: 10px !important;
    color: white !important;
}

input:focus, textarea:focus {
    border-color: #22d3ee !important;
    box-shadow: 0 0 0 2px rgba(34, 211, 238, .15) !important;
}

/* TABS */
button[role="tab"] {
    border-radius: 10px !important;
    transition: .25s ease !important;
}

button[role="tab"][aria-selected="true"] {
    background: linear-gradient(90deg, #164e63, #312e81) !important;
    color: #67e8f9 !important;
}

/* SECTION HEADINGS */
h2, h3 {
    color: #67e8f9 !important;
}

/* FLOATING FOOTER */
.footer {
    text-align: center;
    padding: 25px;
    color: #64748b;
    font-size: 12px;
}

"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="violet",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="ExplainHire | AI Recruitment Platform"
) as demo:

    session = gr.State(None)

    gr.HTML("""
    <div class="hero">
        <div class="logo">✦</div>
        <h1>ExplainHire</h1>
        <p>Where Artificial Intelligence Meets Smarter Recruitment</p>
        <p>✧ Intelligent Screening &nbsp; • &nbsp; Explainable AI &nbsp; • &nbsp; Candidate Insights ✧</p>
    </div>
    """)

    # LOGIN AND REGISTER
    with gr.Column(elem_id="auth-panel", visible=True) as auth_panel:

        gr.Markdown("## Welcome to ExplainHire")
        gr.Markdown("Sign in to continue your journey.")

        with gr.Tabs():

            with gr.Tab("🔐 Login"):

                login_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Recruiter",
                    label="Choose your portal"
                )

                login_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )

                login_password = gr.Textbox(
                    label="Password",
                    type="password",
                    placeholder="Enter your password"
                )

                login_button = gr.Button(
                    "🚀 Enter ExplainHire",
                    variant="primary",
                    size="lg"
                )

                login_status = gr.Markdown()

            with gr.Tab("✨ Create Account"):

                register_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Candidate",
                    label="I am joining as"
                )

                register_name = gr.Textbox(
                    label="Full Name",
                    placeholder="Enter your full name"
                )

                register_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )

                register_password = gr.Textbox(
                    label="Create Password",
                    type="password",
                    placeholder="Minimum 8 characters"
                )

                register_button = gr.Button(
                    "Create My Account",
                    variant="primary"
                )

                register_status = gr.Markdown()

    # RECRUITER DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as recruiter_panel:

        gr.Markdown("""
        # 💼 Recruiter Workspace
        ### Discover the right talent with Explainable AI.
        """)

        recruiter_welcome = gr.Markdown()

        recruiter_files = gr.File(
            label="📂 Upload Candidate Resumes",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        )

        recruiter_jd = gr.Textbox(
            label="📋 Job Description",
            lines=6,
            placeholder="Paste your job description..."
        )

        with gr.Row():
            recruiter_required = gr.Textbox(
                label="Required Skills",
                placeholder="Python, ML, SQL"
            )

            recruiter_preferred = gr.Textbox(
                label="Preferred Skills",
                placeholder="NLP, LLM, AWS"
            )

        recruiter_experience = gr.Number(
            label="Minimum Experience",
            value=0,
            minimum=0
        )

        analyze_button = gr.Button(
            "⚡ Start AI Screening",
            variant="primary",
            size="lg"
        )

        gr.Markdown("## 📊 Candidate Intelligence")

        recruiter_table = gr.Dataframe(
            label="Candidate Ranking"
        )

        recruiter_report = gr.Markdown(
            label="Screening Report"
        )

        recruiter_plot = gr.Plot(
            label="Candidate Comparison"
        )

        recruiter_csv = gr.File(
            label="📥 Download Results"
        )

        gr.Markdown("## 🎙️ AI Voice Explanation")

        voice_candidate = gr.Dropdown(
            choices=[],
            label="Choose Candidate"
        )

        voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Select Explanation Voice"
        )

        voice_button = gr.Button(
            "🔊 Generate Voice Explanation",
            variant="primary"
        )

        voice_output = gr.Audio(
            label="AI Candidate Explanation",
            type="filepath"
        )

        recruiter_logout = gr.Button("🚪 Logout")

    # CANDIDATE DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as candidate_panel:

        gr.Markdown("""
        # 👤 Candidate Portal
        ### Understand your profile. Improve your opportunities.
        """)

        candidate_welcome = gr.Markdown()

        candidate_pdf = gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf"],
            type="filepath"
        )

        candidate_jd = gr.Textbox(
            label="🎯 Target Job Description",
            lines=6,
            placeholder="Paste the job description..."
        )

        candidate_analyze = gr.Button(
            "✨ Analyze My Resume",
            variant="primary",
            size="lg"
        )

        candidate_status = gr.Markdown()

        candidate_skills = gr.Textbox(
            label="🧠 Skills Detected",
            lines=3
        )

        candidate_report = gr.Textbox(
            label="📈 Your Resume Match Report",
            lines=10
        )

        candidate_logout = gr.Button("🚪 Logout")

    gr.HTML("""
    <div class="footer">
        EXPLAINHIRE • AI POWERED RECRUITMENT EXPERIENCE<br>
        Built with Python, Machine Learning and Explainable AI
    </div>
    """)

    # REGISTER
    register_button.click(
        fn=register_user,
        inputs=[
            register_name,
            register_email,
            register_password,
            register_role
        ],
        outputs=register_status
    )

    # LOGIN
    def authenticate(email, password, role):

        user, message = login_user(email, password, role)

        if user is None:
            return (
                None, message,
                gr.update(), gr.update(visible=True),
                gr.update(visible=False),
                gr.update(visible=False),
                gr.update()
            )

        if role == "Recruiter":
            return (
                user, f"✅ {message}",
                gr.update(value=f"### Welcome back, {user['name']}! 👋"),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update()
            )

        return (
            user, f"✅ {message}",
            gr.update(),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(value=f"### Welcome, {user['name']}! 🌟")
        )

    login_button.click(
        fn=authenticate,
        inputs=[login_email, login_password, login_role],
        outputs=[
            session, login_status, recruiter_welcome,
            auth_panel, recruiter_panel, candidate_panel,
            candidate_welcome
        ]
    )

    # RECRUITER SCREENING
    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            recruiter_files,
            recruiter_jd,
            recruiter_required,
            recruiter_preferred,
            recruiter_experience
        ],
        outputs=[
            recruiter_table,
            recruiter_report,
            recruiter_plot,
            recruiter_csv,
            voice_candidate
        ]
    )

    # VOICE
    voice_button.click(
        fn=generate_candidate_voice,
        inputs=[voice_candidate, voice_choice],
        outputs=voice_output
    )

    # CANDIDATE ANALYSIS
    candidate_analyze.click(
        fn=candidate_resume_review,
        inputs=[candidate_pdf, candidate_jd],
        outputs=[
            candidate_status,
            candidate_skills,
            candidate_report
        ]
    )

    # LOGOUT
    def logout_user():
        return (
            None,
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(visible=False)
        )

    recruiter_logout.click(
        fn=logout_user,
        outputs=[
            session,
            auth_panel,
            recruiter_panel,
            candidate_panel
        ]
    )

    candidate_logout.click(
        fn=logout_user,
        outputs=[
            session,
            auth_panel,
            recruiter_panel,
            candidate_panel
        ]
    )

print("ExplainHire interactive UI is ready!")


/tmp/ipykernel_888/2885255409.py:190: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning:

Expected 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1387: UserWarning:

Expected maximum 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.



ExplainHire interactive UI is ready!


In [22]:

demo.launch(
    share=True,
    debug=False
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7bf760d90763dcd839.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:

# CELL 14D — ANIME LOGIN CELEBRATION

anime_css = """

/* Celebration overlay */
#anime-celebration {
    position: fixed !important;
    inset: 0 !important;
    z-index: 99999 !important;
    pointer-events: none !important;
    overflow: hidden !important;
    background: radial-gradient(
        circle,
        rgba(79, 70, 229, .25),
        rgba(5, 10, 30, .88)
    ) !important;
    animation: animeFade 4.5s forwards;
}

.anime-title {
    position: absolute;
    top: 35%;
    width: 100%;
    text-align: center;
    font-size: clamp(30px, 6vw, 65px);
    font-weight: 1000;
    color: white;
    text-shadow:
        0 0 10px #22d3ee,
        0 0 30px #8b5cf6,
        0 0 60px #ec4899;
    animation: animeTitle 3.5s ease-out forwards;
}

.anime-subtitle {
    position: absolute;
    top: 48%;
    width: 100%;
    text-align: center;
    font-size: 20px;
    color: #a5f3fc;
    animation: subtitlePop 3.5s ease-out forwards;
}

.anime-runner {
    position: absolute;
    bottom: 12%;
    left: -150px;
    font-size: 85px;
    animation: cartoonRun 3.8s cubic-bezier(.2,.8,.3,1) forwards;
    filter: drop-shadow(0 0 20px #22d3ee);
}

.anime-runner span {
    display: inline-block;
    animation: runningBounce .18s infinite alternate;
}

.anime-burst {
    position: absolute;
    left: 50%;
    top: 45%;
    width: 20px;
    height: 20px;
    border-radius: 50%;
    background: white;
    box-shadow:
        0 0 20px 15px #22d3ee,
        0 0 60px 40px #8b5cf6,
        0 0 100px 70px #ec4899;
    animation: energyBlast 1.2s ease-out forwards;
}

.anime-star {
    position: absolute;
    font-size: 25px;
    animation: starFly 2.5s ease-out forwards;
}

@keyframes animeFade {
    0%, 75% { opacity: 1; }
    100% { opacity: 0; visibility: hidden; }
}

@keyframes animeTitle {
    0% { opacity: 0; transform: scale(.1) rotate(-8deg); }
    15% { opacity: 1; transform: scale(1.15) rotate(2deg); }
    30%, 75% { opacity: 1; transform: scale(1); }
    100% { opacity: 0; transform: scale(1.5); }
}

@keyframes subtitlePop {
    0%, 15% { opacity: 0; transform: translateY(30px); }
    30%, 75% { opacity: 1; transform: translateY(0); }
    100% { opacity: 0; }
}

@keyframes cartoonRun {
    0% { left: -150px; }
    20% { left: 15%; }
    40% { left: 35%; }
    60% { left: 55%; }
    80% { left: 80%; }
    100% { left: 115%; }
}

@keyframes runningBounce {
    from { transform: translateY(0) rotate(-8deg); }
    to { transform: translateY(-15px) rotate(8deg); }
}

@keyframes energyBlast {
    0% { transform: translate(-50%, -50%) scale(.1); opacity: 1; }
    100% { transform: translate(-50%, -50%) scale(20); opacity: 0; }
}

@keyframes starFly {
    0% { transform: translate(0,0) scale(.2) rotate(0deg); opacity: 1; }
    100% { transform: translate(var(--x), var(--y)) scale(2) rotate(360deg); opacity: 0; }
}

"""

print("Anime celebration animations ready!")


Anime celebration animations ready!


In [24]:

# Add inside the Gradio Blocks

celebration = gr.HTML(
    value="",
    visible=False,
    elem_id="celebration-output"
)


In [25]:

# CELL 14E — CELEBRATION GENERATOR

def anime_welcome(name, role):

    stars = ""

    positions = [
        ("-300px", "-200px"),
        ("300px", "-250px"),
        ("-400px", "100px"),
        ("400px", "150px"),
        ("-200px", "300px"),
        ("250px", "280px"),
        ("100px", "-350px"),
        ("-100px", "-300px")
    ]

    for x, y in positions:
        stars += f"""
        <div class="anime-star"
             style="left:50%;top:45%;
             --x:{x};--y:{y};">
             ✨
        </div>
        """

    return f"""
    <div id="anime-celebration">
        <div class="anime-burst"></div>

        {stars}

        <div class="anime-title">
            ⚡ WELCOME TO EXPLAINHIRE ⚡
        </div>

        <div class="anime-subtitle">
            Hello {name}! Your {role} adventure begins!
        </div>

        <div class="anime-runner">
            <span>🏃‍♂️💨</span>
        </div>
    </div>
    """


In [27]:

login_button.click(
    fn=authenticate,
    inputs=[
        login_email,
        login_password,
        login_role
    ],
    outputs=[
        session,
        login_status,
        recruiter_welcome,
        auth_panel,
        recruiter_panel,
        candidate_panel,
        candidate_welcome,
        celebration
    ]
)


AttributeError: Cannot call click outside of a gradio.Blocks context.

In [39]:
import gradio as gr

# 1. Combine Custom CSS with Anime Celebration Animation CSS
complete_css = custom_css + anime_css

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="violet",
        neutral_hue="slate"
    ),
    css=complete_css,
    title="ExplainHire | AI Recruitment Platform"
) as demo:

    session = gr.State(None)

    # 2. Add the Celebration HTML Placeholder inside the Blocks Context
    celebration = gr.HTML(
        value="",
        visible=False,
        elem_id="celebration-output"
    )

    gr.HTML("""
    <div class="hero">
        <div class="logo">✦</div>
        <h1>ExplainHire</h1>
        <p>Where Artificial Intelligence Meets Smarter Recruitment</p>
        <p>✧ Intelligent Screening &nbsp; • &nbsp; Explainable AI &nbsp; • &nbsp; Candidate Insights ✧</p>
    </div>
    """)

    # LOGIN AND REGISTER PANEL
    with gr.Column(elem_id="auth-panel", visible=True) as auth_panel:
        gr.Markdown("## Welcome to ExplainHire")
        gr.Markdown("Sign in to continue your journey.")

        with gr.Tabs():
            with gr.Tab("🔐 Login"):
                login_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Recruiter",
                    label="Choose your portal"
                )
                login_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )
                login_password = gr.Textbox(
                    label="Password",
                    type="password",
                    placeholder="Enter your password"
                )
                login_button = gr.Button(
                    "🚀 Enter ExplainHire",
                    variant="primary",
                    size="lg"
                )
                login_status = gr.Markdown()

            with gr.Tab("✨ Create Account"):
                register_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Candidate",
                    label="I am joining as"
                )
                register_name = gr.Textbox(
                    label="Full Name",
                    placeholder="Enter your full name"
                )
                register_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )
                register_password = gr.Textbox(
                    label="Create Password",
                    type="password",
                    placeholder="Minimum 8 characters"
                )
                register_button = gr.Button(
                    "Create My Account",
                    variant="primary"
                )
                register_status = gr.Markdown()

    # RECRUITER DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as recruiter_panel:
        gr.Markdown("""
        # 💼 Recruiter Workspace
        ### Discover the right talent with Explainable AI.
        """)
        recruiter_welcome = gr.Markdown()

        recruiter_files = gr.File(
            label="📂 Upload Candidate Resumes",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        )

        recruiter_jd = gr.Textbox(
            label="📋 Job Description",
            lines=6,
            placeholder="Paste your job description..."
        )

        with gr.Row():
            recruiter_required = gr.Textbox(
                label="Required Skills",
                placeholder="Python, ML, SQL"
            )
            recruiter_preferred = gr.Textbox(
                label="Preferred Skills",
                placeholder="NLP, LLM, AWS"
            )

        recruiter_experience = gr.Number(
            label="Minimum Experience",
            value=0,
            minimum=0
        )

        analyze_button = gr.Button(
            "⚡ Start AI Screening",
            variant="primary",
            size="lg"
        )

        gr.Markdown("## 📊 Candidate Intelligence")
        recruiter_table = gr.Dataframe(label="Candidate Ranking")
        recruiter_report = gr.Markdown(label="Screening Report")
        recruiter_plot = gr.Plot(label="Candidate Comparison")
        recruiter_csv = gr.File(label="📥 Download Results")

        gr.Markdown("## 🎙️ AI Voice Explanation")
        voice_candidate = gr.Dropdown(choices=[], label="Choose Candidate")
        voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Select Explanation Voice"
        )
        voice_button = gr.Button(
            "🔊 Generate Voice Explanation",
            variant="primary"
        )
        voice_output = gr.Audio(
            label="AI Candidate Explanation",
            type="filepath"
        )
        recruiter_logout = gr.Button("🚪 Logout")

    # CANDIDATE DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as candidate_panel:
        gr.Markdown("""
        # 👤 Candidate Portal
        ### Understand your profile. Improve your opportunities.
        """)
        candidate_welcome = gr.Markdown()

        candidate_pdf = gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf"],
            type="filepath"
        )
        candidate_jd = gr.Textbox(
            label="🎯 Target Job Description",
            lines=6,
            placeholder="Paste the job description..."
        )
        candidate_analyze = gr.Button(
            "✨ Analyze My Resume",
            variant="primary",
            size="lg"
        )
        candidate_status = gr.Markdown()
        candidate_skills = gr.Textbox(label="🧠 Skills Detected", lines=3)
        candidate_report = gr.Textbox(label="📈 Your Resume Match Report", lines=10)
        candidate_logout = gr.Button("🚪 Logout")

    gr.HTML("""
    <div class="footer">
        EXPLAINHIRE • AI POWERED RECRUITMENT EXPERIENCE<br>
        Built with Python, Machine Learning and Explainable AI
    </div>
    """)

    # 3. Define the updated authenticate function that returns the anime welcome page
    def authenticate(email, password, role):
        user, message = login_user(email, password, role)

        if user is None:
            return (
                None, message,
                gr.update(), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False),
                gr.update(), gr.update(value="", visible=False)
            )

        # Generate animation HTML content
        celebration_html = anime_welcome(user["name"], user["role"])

        if role == "Recruiter":
            return (
                user, f"✅ {message}",
                gr.update(value=f"### Welcome back, {user['name']}! 👋"),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(),
                gr.update(value=celebration_html, visible=True)
            )

        return (
            user, f"✅ {message}",
            gr.update(),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(value=f"### Welcome, {user['name']}! 🌟"),
            gr.update(value=celebration_html, visible=True)
        )

    # Register actions inside the block
    register_button.click(
        fn=register_user,
        inputs=[register_name, register_email, register_password, register_role],
        outputs=register_status
    )

    login_button.click(
        fn=authenticate,
        inputs=[login_email, login_password, login_role],
        outputs=[
            session, login_status, recruiter_welcome,
            auth_panel, recruiter_panel, candidate_panel,
            candidate_welcome, celebration
        ]
    )

    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            recruiter_files,
            recruiter_jd,
            recruiter_required,
            recruiter_preferred,
            recruiter_experience
        ],
        outputs=[
            recruiter_table,
            recruiter_report,
            recruiter_plot,
            recruiter_csv,
            voice_candidate
        ]
    )

    voice_button.click(
        fn=generate_candidate_voice,
        inputs=[voice_candidate, voice_choice],
        outputs=voice_output
    )

    candidate_analyze.click(
        fn=candidate_resume_review,
        inputs=[candidate_pdf, candidate_jd],
        outputs=[candidate_status, candidate_skills, candidate_report]
    )

    def logout_user():
        return (
            None, gr.update(visible=True),
            gr.update(visible=False), gr.update(visible=False)
        )

    recruiter_logout.click(
        fn=logout_user,
        outputs=[session, auth_panel, recruiter_panel, candidate_panel]
    )

    candidate_logout.click(
        fn=logout_user,
        outputs=[session, auth_panel, recruiter_panel, candidate_panel]
    )

print("Unified ExplainHire App with integrated animation flow configured!")

/tmp/ipykernel_888/2720460269.py:6: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning:

Expected 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1387: UserWarning:

Expected maximum 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.



Unified ExplainHire App with integrated animation flow configured!


In [40]:
demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b21d8f4861fd08ba22.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [30]:


css=custom_css + anime_css



In [31]:

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="violet",
        neutral_hue="slate"
    ),
    css=custom_css + anime_css,
    title="ExplainHire"
) as demo:
    # Keep your existing application components here
    pass


/tmp/ipykernel_888/3321992057.py:1: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



In [32]:

!pip install edge-tts


In [33]:

# CELL 14F — CANDIDATE AI VOICE ASSISTANT

import edge_tts
import uuid

async def candidate_voice_assistant(skills, report, voice_choice):

    if not report or not report.strip():
        raise gr.Error("Please analyze your resume first.")

    voice_map = {
        "Male": "en-IN-PrabhatNeural",
        "Female": "en-IN-NeerjaNeural"
    }

    voice = voice_map.get(
        voice_choice,
        "en-IN-NeerjaNeural"
    )

    narration = f"""
    Hello! Welcome to your ExplainHire candidate assistant.

    Let's review your resume.

    Your detected skills are:
    {skills if skills else "No skills were detected."}

    Here is your resume analysis.

    {report}

    Keep improving your technical skills,
    build practical projects and update your resume regularly.

    Thank you for using ExplainHire.
    All the best for your career!
    """

    filename = f"/content/candidate_assistant_{uuid.uuid4().hex}.mp3"

    communicator = edge_tts.Communicate(
        narration,
        voice
    )

    await communicator.save(filename)

    return filename


print("Candidate AI Voice Assistant is ready!")


Candidate AI Voice Assistant is ready!


In [34]:
candidate_report = gr.Textbox(
    label="📈 Your Resume Match Report",
    lines=10
)

In [35]:

# CANDIDATE VOICE ASSISTANT UI

gr.Markdown("""
## 🤖 Your Personal AI Career Assistant

Listen to your resume analysis and understand
your strengths and areas for improvement.
""")

candidate_voice_choice = gr.Radio(
    ["Male", "Female"],
    value="Female",
    label="Choose Assistant Voice"
)

candidate_voice_button = gr.Button(
    "🎙️ Talk to My AI Assistant",
    variant="primary",
    size="lg"
)

candidate_voice_output = gr.Audio(
    label="Your AI Assistant",
    type="filepath"
)


In [41]:
import gradio as gr

# Combine Styles
complete_css = custom_css + anime_css

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="violet",
        neutral_hue="slate"
    ),
    css=complete_css,
    title="ExplainHire | AI Recruitment Platform"
) as demo:

    session = gr.State(None)

    # Celebration HTML Layer
    celebration = gr.HTML(
        value="",
        visible=False,
        elem_id="celebration-output"
    )

    gr.HTML("""
    <div class="hero">
        <div class="logo">✦</div>
        <h1>ExplainHire</h1>
        <p>Where Artificial Intelligence Meets Smarter Recruitment</p>
        <p>✧ Intelligent Screening &nbsp; • &nbsp; Explainable AI &nbsp; • &nbsp; Candidate Insights ✧</p>
    </div>
    """)

    # LOGIN AND REGISTER PANEL
    with gr.Column(elem_id="auth-panel", visible=True) as auth_panel:
        gr.Markdown("## Welcome to ExplainHire")
        gr.Markdown("Sign in to continue your journey.")

        with gr.Tabs():
            with gr.Tab("🔐 Login"):
                login_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Recruiter",
                    label="Choose your portal"
                )
                login_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )
                login_password = gr.Textbox(
                    label="Password",
                    type="password",
                    placeholder="Enter your password"
                )
                login_button = gr.Button(
                    "🚀 Enter ExplainHire",
                    variant="primary",
                    size="lg"
                )
                login_status = gr.Markdown()

            with gr.Tab("✨ Create Account"):
                register_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Candidate",
                    label="I am joining as"
                )
                register_name = gr.Textbox(
                    label="Full Name",
                    placeholder="Enter your full name"
                )
                register_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )
                register_password = gr.Textbox(
                    label="Create Password",
                    type="password",
                    placeholder="Minimum 8 characters"
                )
                register_button = gr.Button(
                    "Create My Account",
                    variant="primary"
                )
                register_status = gr.Markdown()

    # RECRUITER DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as recruiter_panel:
        gr.Markdown("""
        # 💼 Recruiter Workspace
        ### Discover the right talent with Explainable AI.
        """)
        recruiter_welcome = gr.Markdown()

        recruiter_files = gr.File(
            label="📂 Upload Candidate Resumes",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        )

        recruiter_jd = gr.Textbox(
            label="📋 Job Description",
            lines=6,
            placeholder="Paste your job description..."
        )

        with gr.Row():
            recruiter_required = gr.Textbox(
                label="Required Skills",
                placeholder="Python, ML, SQL"
            )
            recruiter_preferred = gr.Textbox(
                label="Preferred Skills",
                placeholder="NLP, LLM, AWS"
            )

        recruiter_experience = gr.Number(
            label="Minimum Experience",
            value=0,
            minimum=0
        )

        analyze_button = gr.Button(
            "⚡ Start AI Screening",
            variant="primary",
            size="lg"
        )

        gr.Markdown("## 📊 Candidate Intelligence")
        recruiter_table = gr.Dataframe(label="Candidate Ranking")
        recruiter_report = gr.Markdown(label="Screening Report")
        recruiter_plot = gr.Plot(label="Candidate Comparison")
        recruiter_csv = gr.File(label="📥 Download Results")

        gr.Markdown("## 🎙️ AI Voice Explanation")
        voice_candidate = gr.Dropdown(choices=[], label="Choose Candidate")
        voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Select Explanation Voice"
        )
        voice_button = gr.Button(
            "🔊 Generate Voice Explanation",
            variant="primary"
        )
        voice_output = gr.Audio(
            label="AI Candidate Explanation",
            type="filepath"
        )
        recruiter_logout = gr.Button("🚪 Logout")

    # CANDIDATE DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as candidate_panel:
        gr.Markdown("""
        # 👤 Candidate Portal
        ### Understand your profile. Improve your opportunities.
        """)
        candidate_welcome = gr.Markdown()

        candidate_pdf = gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf"],
            type="filepath"
        )
        candidate_jd = gr.Textbox(
            label="🎯 Target Job Description",
            lines=6,
            placeholder="Paste the job description..."
        )
        candidate_analyze = gr.Button(
            "✨ Analyze My Resume",
            variant="primary",
            size="lg"
        )
        candidate_status = gr.Markdown()
        candidate_skills = gr.Textbox(label="🧠 Skills Detected", lines=3)
        candidate_report = gr.Textbox(label="📈 Your Resume Match Report", lines=10)

        # INTEGRATE CANDIDATE VOICE ASSISTANT IN THE PANEL
        gr.Markdown("""
        ## 🤖 Your Personal AI Career Assistant
        Listen to your resume analysis and understand your strengths and areas for improvement.
        """)
        candidate_voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Choose Assistant Voice"
        )
        candidate_voice_button = gr.Button(
            "🎙️ Talk to My AI Assistant",
            variant="primary",
            size="lg"
        )
        candidate_voice_output = gr.Audio(
            label="Your AI Assistant",
            type="filepath"
        )

        candidate_logout = gr.Button("🚪 Logout")

    gr.HTML("""
    <div class="footer">
        EXPLAINHIRE • AI POWERED RECRUITMENT EXPERIENCE<br>
        Built with Python, Machine Learning and Explainable AI
    </div>
    """)

    # Authenticate Function
    def authenticate(email, password, role):
        user, message = login_user(email, password, role)

        if user is None:
            return (
                None, message,
                gr.update(), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False),
                gr.update(), gr.update(value="", visible=False)
            )

        celebration_html = anime_welcome(user["name"], user["role"])

        if role == "Recruiter":
            return (
                user, f"✅ {message}",
                gr.update(value=f"### Welcome back, {user['name']}! 👋"),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(),
                gr.update(value=celebration_html, visible=True)
            )

        return (
            user, f"✅ {message}",
            gr.update(),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(value=f"### Welcome, {user['name']}! 🌟"),
            gr.update(value=celebration_html, visible=True)
        )

    # Click Event Bindings
    register_button.click(
        fn=register_user,
        inputs=[register_name, register_email, register_password, register_role],
        outputs=register_status
    )

    login_button.click(
        fn=authenticate,
        inputs=[login_email, login_password, login_role],
        outputs=[
            session, login_status, recruiter_welcome,
            auth_panel, recruiter_panel, candidate_panel,
            candidate_welcome, celebration
        ]
    )

    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            recruiter_files,
            recruiter_jd,
            recruiter_required,
            recruiter_preferred,
            recruiter_experience
        ],
        outputs=[
            recruiter_table,
            recruiter_report,
            recruiter_plot,
            recruiter_csv,
            voice_candidate
        ]
    )

    voice_button.click(
        fn=generate_candidate_voice,
        inputs=[voice_candidate, voice_choice],
        outputs=voice_output
    )

    candidate_analyze.click(
        fn=candidate_resume_review,
        inputs=[candidate_pdf, candidate_jd],
        outputs=[candidate_status, candidate_skills, candidate_report]
    )

    # Connect Candidate Assistant Event
    candidate_voice_button.click(
        fn=candidate_voice_assistant,
        inputs=[
            candidate_skills,
            candidate_report,
            candidate_voice_choice
        ],
        outputs=candidate_voice_output
    )

    def logout_user():
        return (
            None, gr.update(visible=True),
            gr.update(visible=False), gr.update(visible=False)
        )

    recruiter_logout.click(
        fn=logout_user,
        outputs=[session, auth_panel, recruiter_panel, candidate_panel]
    )

    candidate_logout.click(
        fn=logout_user,
        outputs=[session, auth_panel, recruiter_panel, candidate_panel]
    )

print("Secure ExplainHire App configured successfully within context!")

Secure ExplainHire App configured successfully within context!


/tmp/ipykernel_888/4012921599.py:6: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning:

Expected 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1387: UserWarning:

Expected maximum 3 arguments for function <function analyze_candidates at 0x79cd2216be20>, received 5.



In [42]:
demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb1d955ee0eedb6617.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [45]:
import os
import pandas as pd
import plotly.express as px
import gradio as gr

def analyze_candidates(files, job_description, required_skills, preferred_skills, min_experience):
    global candidate_results

    if not files:
        raise gr.Error("Please upload PDF resumes.")

    if not job_description.strip():
        raise gr.Error("Please enter a job description.")

    results = []

    for file_path in files:
        resume_text = extract_pdf_text(file_path)

        if len(resume_text.strip()) < 30:
            continue

        # Combine explicit required skills string with the JD text to ensure proper matching
        combined_jd = job_description
        if required_skills.strip():
            combined_jd += " " + required_skills

        analysis = screen_resume(
            resume_text,
            combined_jd,
            preferred_skills
        )

        # Apply experience filter
        candidate_exp = analysis.get("experience_years", 0)
        if candidate_exp < min_experience:
            analysis["score"] = max(0.0, analysis["score"] - 25.0)  # Moderate penalty for insufficient experience
            analysis["recommendation"] = "Below Experience Req."

        results.append({
            "Candidate": os.path.basename(file_path),
            "Match Score": analysis["score"],
            "Category": analysis["category"],
            "Confidence": analysis["confidence"],
            "Required Skills": analysis["required"],
            "Preferred Skills": analysis["preferred"],
            "Similarity": analysis["similarity"],
            "Experience": analysis["experience"],
            "Education": analysis["education"],
            "Recommendation": analysis["recommendation"],
            "Analysis": analysis
        })

    if not results:
        raise gr.Error("No readable resumes found.")

    results.sort(
        key=lambda x: x["Match Score"],
        reverse=True
    )

    candidate_results = results

    table = pd.DataFrame([
        {k: v for k, v in r.items() if k != "Analysis"}
        for r in results
    ])

    report = "# EXPLAINHIRE - CANDIDATE REPORT\n"
    for i, r in enumerate(results, 1):
        a = r["Analysis"]
        report += f"""\n---\n## {i}. {r['Candidate']}\n\n**Score:** {a['score']}/100\n\n**Category:** {a['category']}\n\n**Recommendation:** {a['recommendation']}\n\n**Matched Skills:** {', '.join(a['matched']) or 'None'}\n\n**Missing Skills:** {', '.join(a['missing']) or 'None'}\n\n**Experience:** {a['experience_years']} years\n"""

    export_path = "/content/ExplainHire_Ranking.csv"
    table.to_csv(export_path, index=False)

    fig = px.bar(
        table,
        x="Candidate",
        y="Match Score",
        color="Match Score",
        range_y=[0, 100],
        title="Candidate Ranking"
    )
    fig.update_layout(template="plotly_dark")

    choices = [r["Candidate"] for r in results]

    return (
        table,
        report,
        fig,
        export_path,
        gr.update(choices=choices, value=choices[0] if choices else None)
    )

print("Backend candidate analysis function fixed and updated with explicit imports!")

Backend candidate analysis function fixed and updated with explicit imports!


In [46]:
import os
import pandas as pd
import plotly.express as px
import gradio as gr

complete_css = custom_css + anime_css

with gr.Blocks(
    title="ExplainHire | AI Recruitment Platform"
) as demo:

    session = gr.State(None)

    celebration = gr.HTML(
        value="",
        visible=False,
        elem_id="celebration-output"
    )

    gr.HTML("""
    <div class="hero">
        <div class="logo">✦</div>
        <h1>ExplainHire</h1>
        <p>Where Artificial Intelligence Meets Smarter Recruitment</p>
        <p>✧ Intelligent Screening &nbsp; • &nbsp; Explainable AI &nbsp; • &nbsp; Candidate Insights ✧</p>
    </div>
    """)

    # LOGIN AND REGISTER PANEL
    with gr.Column(elem_id="auth-panel", visible=True) as auth_panel:
        gr.Markdown("## Welcome to ExplainHire")
        gr.Markdown("Sign in to continue your journey.")

        with gr.Tabs():
            with gr.Tab("🔐 Login"):
                login_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Recruiter",
                    label="Choose your portal"
                )
                login_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )
                login_password = gr.Textbox(
                    label="Password",
                    type="password",
                    placeholder="Enter your password"
                )
                login_button = gr.Button(
                    "🚀 Enter ExplainHire",
                    variant="primary",
                    size="lg"
                )
                login_status = gr.Markdown()

            with gr.Tab("✨ Create Account"):
                register_role = gr.Radio(
                    ["Recruiter", "Candidate"],
                    value="Candidate",
                    label="I am joining as"
                )
                register_name = gr.Textbox(
                    label="Full Name",
                    placeholder="Enter your full name"
                )
                register_email = gr.Textbox(
                    label="Email Address",
                    placeholder="you@example.com"
                )
                register_password = gr.Textbox(
                    label="Create Password",
                    type="password",
                    placeholder="Minimum 8 characters"
                )
                register_button = gr.Button(
                    "Create My Account",
                    variant="primary"
                )
                register_status = gr.Markdown()

    # RECRUITER DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as recruiter_panel:
        gr.Markdown("""
        # 💼 Recruiter Workspace
        ### Discover the right talent with Explainable AI.
        """)
        recruiter_welcome = gr.Markdown()

        recruiter_files = gr.File(
            label="📂 Upload Candidate Resumes",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        )

        recruiter_jd = gr.Textbox(
            label="📋 Job Description",
            lines=6,
            placeholder="Paste your job description..."
        )

        with gr.Row():
            recruiter_required = gr.Textbox(
                label="Required Skills",
                placeholder="Python, ML, SQL"
            )
            recruiter_preferred = gr.Textbox(
                label="Preferred Skills",
                placeholder="NLP, LLM, AWS"
            )

        recruiter_experience = gr.Number(
            label="Minimum Experience",
            value=0,
            minimum=0
        )

        analyze_button = gr.Button(
            "⚡ Start AI Screening",
            variant="primary",
            size="lg"
        )

        gr.Markdown("## 📊 Candidate Intelligence")
        recruiter_table = gr.Dataframe(label="Candidate Ranking")
        recruiter_report = gr.Markdown(label="Screening Report")
        recruiter_plot = gr.Plot(label="Candidate Comparison")
        recruiter_csv = gr.File(label="📥 Download Results")

        gr.Markdown("## 🎙️ AI Voice Explanation")
        voice_candidate = gr.Dropdown(choices=[], label="Choose Candidate")
        voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Select Explanation Voice"
        )
        voice_button = gr.Button(
            "🔊 Generate Voice Explanation",
            variant="primary"
        )
        voice_output = gr.Audio(
            label="AI Candidate Explanation",
            type="filepath"
        )
        recruiter_logout = gr.Button("🚪 Logout")

    # CANDIDATE DASHBOARD
    with gr.Column(visible=False, elem_classes=["dashboard"]) as candidate_panel:
        gr.Markdown("""
        # 👤 Candidate Portal
        ### Understand your profile. Improve your opportunities.
        """)
        candidate_welcome = gr.Markdown()

        candidate_pdf = gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf"],
            type="filepath"
        )
        candidate_jd = gr.Textbox(
            label="🎯 Target Job Description",
            lines=6,
            placeholder="Paste the job description..."
        )
        candidate_analyze = gr.Button(
            "✨ Analyze My Resume",
            variant="primary",
            size="lg"
        )
        candidate_status = gr.Markdown()
        candidate_skills = gr.Textbox(label="🧠 Skills Detected", lines=3)
        candidate_report = gr.Textbox(label="📈 Your Resume Match Report", lines=10)

        gr.Markdown("""
        ## 🤖 Your Personal AI Career Assistant
        Listen to your resume analysis and understand your strengths and areas for improvement.
        """)
        candidate_voice_choice = gr.Radio(
            ["Male", "Female"],
            value="Female",
            label="Choose Assistant Voice"
        )
        candidate_voice_button = gr.Button(
            "🎙️ Talk to My AI Assistant",
            variant="primary",
            size="lg"
        )
        candidate_voice_output = gr.Audio(
            label="Your AI Assistant",
            type="filepath"
        )
        candidate_logout = gr.Button("🚪 Logout")

    gr.HTML("""
    <div class="footer">
        EXPLAINHIRE • AI POWERED RECRUITMENT EXPERIENCE<br>
        Built with Python, Machine Learning and Explainable AI
    </div>
    """)

    def authenticate(email, password, role):
        user, message = login_user(email, password, role)

        if user is None:
            return (
                None, message,
                gr.update(), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False),
                gr.update(), gr.update(value="", visible=False)
            )

        celebration_html = anime_welcome(user["name"], user["role"])

        if role == "Recruiter":
            return (
                user, f"✅ {message}",
                gr.update(value=f"### Welcome back, {user['name']}! 👋"),
                gr.update(visible=False),
                gr.update(visible=True),
                gr.update(visible=False),
                gr.update(),
                gr.update(value=celebration_html, visible=True)
            )

        return (
            user, f"✅ {message}",
            gr.update(),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(value=f"### Welcome, {user['name']}! 🌟"),
            gr.update(value=celebration_html, visible=True)
        )

    register_button.click(
        fn=register_user,
        inputs=[register_name, register_email, register_password, register_role],
        outputs=register_status
    )

    login_button.click(
        fn=authenticate,
        inputs=[login_email, login_password, login_role],
        outputs=[
            session, login_status, recruiter_welcome,
            auth_panel, recruiter_panel, candidate_panel,
            candidate_welcome, celebration
        ]
    )

    analyze_button.click(
        fn=analyze_candidates,
        inputs=[
            recruiter_files,
            recruiter_jd,
            recruiter_required,
            recruiter_preferred,
            recruiter_experience
        ],
        outputs=[
            recruiter_table,
            recruiter_report,
            recruiter_plot,
            recruiter_csv,
            voice_candidate
        ]
    )

    voice_button.click(
        fn=generate_candidate_voice,
        inputs=[voice_candidate, voice_choice],
        outputs=voice_output
    )

    candidate_analyze.click(
        fn=candidate_resume_review,
        inputs=[candidate_pdf, candidate_jd],
        outputs=[candidate_status, candidate_skills, candidate_report]
    )

    candidate_voice_button.click(
        fn=candidate_voice_assistant,
        inputs=[
            candidate_skills,
            candidate_report,
            candidate_voice_choice
        ],
        outputs=candidate_voice_output
    )

    def logout_user():
        return (
            None, gr.update(visible=True),
            gr.update(visible=False), gr.update(visible=False)
        )

    recruiter_logout.click(
        fn=logout_user,
        outputs=[session, auth_panel, recruiter_panel, candidate_panel]
    )

    candidate_logout.click(
        fn=logout_user,
        outputs=[session, auth_panel, recruiter_panel, candidate_panel]
    )

demo.launch(
    share=True,
    debug=False,
    theme=gr.themes.Soft(primary_hue="cyan", secondary_hue="violet", neutral_hue="slate"),
    css=complete_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e84107d937db054932.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
